In [ ]:
#@title 1. Options
REPO = "university-of-toronto-humanoid-robotics/humanoid" #@param {type:"string"}
BRANCH = "kartik/locomotion" #@param {type:"string"}
TASK = "G1-Flat-HeightMap" #@param {type:"string"}
NUM_ENVS = 4096 #@param {type:"integer"}
ITERATIONS = 2000 #@param {type:"integer"}
ROLLOUT_STEPS = 24 #@param {type:"integer"}
NUM_MINIBATCHES = 4 #@param {type:"integer"}
SAVE_INTERVAL = 200 #@param {type:"integer"}
RUN_NAME = "flat_map_test" #@param {type:"string"}
WANDB_PROJECT = "humanoid-locomotion" #@param {type:"string"}
WANDB_ENTITY = "" #@param {type:"string"}
RESUME_CHECKPOINT = "" #@param {type:"string"}

from pathlib import Path
import os
import subprocess
import sys

REPO_DIR = Path("/content/humanoid")
PYTHON = "/content/mjlab-venv/bin/python"
assert all(n > 0 for n in (NUM_ENVS, ITERATIONS, ROLLOUT_STEPS, NUM_MINIBATCHES, SAVE_INTERVAL))
assert (NUM_ENVS * ROLLOUT_STEPS) % NUM_MINIBATCHES == 0
print(f"{NUM_ENVS:,} robots; {ITERATIONS:,} updates; {NUM_ENVS * ROLLOUT_STEPS:,} transitions/update")


In [ ]:
# 2. Sign in through your browser, then clone your branch and submodule.
import shutil
from IPython.display import HTML, display

if shutil.which("gh") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "gh"], check=True)
git_env = {**os.environ, "GH_BROWSER": "echo", "GH_PROMPT_DISABLED": "1", "GIT_TERMINAL_PROMPT": "0"}
git_env.pop("GH_TOKEN", None)
git_env.pop("GITHUB_TOKEN", None)
if subprocess.run(["gh", "auth", "status", "--hostname", "github.com"], env=git_env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode:
    display(HTML('<a href="https://github.com/login/device" target="_blank">Open GitHub authorization</a><p>Enter the code below and approve access. This cell waits for you.</p>'))
    login = subprocess.Popen(["gh", "auth", "login", "--web", "--hostname", "github.com", "--git-protocol", "https"], env=git_env, stdin=subprocess.DEVNULL, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        for line in login.stdout:
            print(line, end="", flush=True)
        if login.wait():
            raise RuntimeError("GitHub login failed. Check the output above.")
    except KeyboardInterrupt:
        login.send_signal(2)
        login.wait()
        raise
subprocess.run(["gh", "auth", "setup-git", "--hostname", "github.com"], env=git_env, check=True)
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, f"https://github.com/{REPO}.git", str(REPO_DIR)], env=git_env, check=True)
else:
    current = subprocess.check_output(["git", "-C", str(REPO_DIR), "branch", "--show-current"], text=True).strip()
    assert current == BRANCH, f"Existing checkout is on {current}; use a fresh runtime to change branches."
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], env=git_env, check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "submodule", "update", "--init", "--recursive", "training/unitree_rl_mjlab"], env=git_env, check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "log", "-1", "--oneline"], check=True)


In [ ]:
# 3. Install in an isolated venv using the same pins as the team's Docker setup.
# The requirements filename says native, but pip selects Linux/CUDA wheels here.
assert (3, 10) <= sys.version_info[:2] < (3, 14), "Use Python 3.10–3.13 with these pins."
if not Path(PYTHON).exists():
    subprocess.run([sys.executable, "-m", "venv", "/content/mjlab-venv"], check=True)
subprocess.run([PYTHON, "-m", "pip", "install", "-r", str(REPO_DIR / "docs/requirements-native-mjlab.txt")], check=True)


In [ ]:
# 4. Mount Drive and verify CUDA for both PyTorch and Warp.
from google.colab import drive
drive.mount("/content/drive")
WORK_DIR = Path("/content/drive/MyDrive/humanoid")
WORK_DIR.mkdir(parents=True, exist_ok=True)
subprocess.run(["nvidia-smi"], check=True)
subprocess.run([PYTHON, "-c", "import torch, warp as wp; wp.init(); assert torch.cuda.is_available() and wp.is_cuda_available(), 'CUDA unavailable'; print(torch.cuda.get_device_name(0)); wp.zeros(1, device='cuda:0'); wp.synchronize_device('cuda:0')"], check=True)
print("Outputs will be saved under:", WORK_DIR / "logs/rsl_rl")


In [ ]:
# 5. Train with W&B logging. Use Colab's stop button to interrupt.
from google.colab import userdata
train_env = {
    **os.environ, "WANDB_API_KEY": userdata.get("WANDB_TOKEN"), "WANDB_MODE": "online",
    "WANDB_X_DISABLE_STATS": "false", "WANDB_X_STATS_SAMPLING_INTERVAL": "5",
}
if WANDB_ENTITY:
    train_env["WANDB_USERNAME"] = WANDB_ENTITY  # The installed RSL-RL writer reads this name.
command = [
    PYTHON, "-u", str(REPO_DIR / "training/scripts/train.py"), TASK,
    "--gpu-ids", "[0]",
    "--env.scene.num-envs", str(NUM_ENVS),
    "--agent.max-iterations", str(ITERATIONS),
    "--agent.num-steps-per-env", str(ROLLOUT_STEPS),
    "--agent.algorithm.num-mini-batches", str(NUM_MINIBATCHES),
    "--agent.save-interval", str(SAVE_INTERVAL),
    "--agent.logger", "wandb",
    "--agent.wandb-project", WANDB_PROJECT,
    "--agent.run-name", RUN_NAME,
]
if RESUME_CHECKPOINT:
    assert Path(RESUME_CHECKPOINT).is_file(), "Resume checkpoint not found."
    command += ["--from", RESUME_CHECKPOINT]
process = subprocess.Popen(command, cwd=WORK_DIR, env=train_env)
try:
    return_code = process.wait()
except KeyboardInterrupt:
    process.send_signal(2)  # Forward Colab's interrupt to the trainer.
    process.wait()
    raise
if return_code:
    raise subprocess.CalledProcessError(return_code, command)


In [ ]:
# 6. Find recent checkpoints. Copy a full path into RESUME_CHECKPOINT to continue.
checkpoints = sorted(WORK_DIR.glob("logs/rsl_rl/*/*/model_*.pt"), key=lambda p: p.stat().st_mtime)
for checkpoint in checkpoints[-10:]:
    print(checkpoint)
print("Each run folder also contains TensorBoard logs, policy.onnx, and params/ configs. Cloud metrics are in your W&B project.")
